In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
from transformers import TextStreamer
from tqdm.auto import tqdm

In [3]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    torch_dtype=torch.bfloat16
)

temperature = 1.0  # You can adjust this value (e.g., 0.7 for more focused, 1.2 for more diverse)
input_text = "Write me an introduction of a Loan Agreement between two parties. The agreement should include realistic names of the parties, loan amount, interest rate, repayment schedule, and dates."
input_ids = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(**input_ids, max_new_tokens=200, temperature=temperature)
print(tokenizer.decode(outputs[0]))

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

<bos>Write me an introduction of a Loan Agreement between two parties.

**Loan Agreement**

**Parties:**

* **Borrower:** [Name of borrower]
* **Lender:** [Name of lender]

**Date:** [Date]

**Loan Details:**

* **Amount:** [Amount of loan]
* **Interest rate:** [Interest rate]% per annum
* **Loan term:** [Loan term] years
* **Repayment terms:** [Repayment terms]

**Conditions:**

* **Collateral:** [Description of collateral provided]
* **Insurance:** [Description of insurance provided]
* **Default provisions:** [Description of default provisions]

**Agreement:**

In consideration of the mutual covenants and agreements contained herein, the parties agree as follows:

**1. Loan Disbursement**

* The lender shall lend the borrower the specified amount of [Amount of loan] on [Date].
* The borrower shall repay the loan principal and interest payments in accordance with the repayment terms specified in section 


In [7]:
from faker import Faker

fake = Faker()

In [9]:
address = fake.address().replace('\n', ', ')
date = fake.date()
email = fake.email()

input_text = f"Write me an introduction of a Loan Agreement between two companies. The agreement should include realistic names of the parties (companies), loan amount, interest rate, repayment schedule, dates (example: {date}), locations (example: {address}), and contact information (example: {email}). Output exactly one paragraph of text. The entire response must be a continuous block with no line breaks."
input_ids = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(**input_ids, max_new_tokens=200)
print(tokenizer.decode(outputs[0]))

<bos>Write me an introduction of a Loan Agreement between two companies. The agreement should include realistic names of the parties (companies), loan amount, interest rate, repayment schedule, dates (example: 2015-07-26), locations (example: 072 Wagner Forks, Franciscotown, PW 03171), and contact information (example: mark53@example.com). Output exactly one paragraph of text. The entire response must be a continuous block with no line breaks.

**Loan Agreement**

**Parties:**

* **ABC Company**
* **XYZ Company**

**Loan Amount:** $100,000

**Interest Rate:** 10% per annum

**Repayment Schedule:**

* Principal repayment: $50,000 by 2016-07-26
* Interest payments: $25,000 per annum, starting from 2016-08-01

**Dates:**

* Loan inception: 2015-07-26
* First principal repayment: 2016-07-26
* Final principal repayment: 2018-07-26

**Locations:**

* 072 Wagner Forks, Franciscotown, PW 03171

**Contact Information:**

* Mark53@example.com

This Loan Agreement, made


In [67]:
from faker import Faker

fake = Faker()

lender = fake.company()
lender_address = fake.address().replace('\n', ', ')
borrower = fake.company()
borrower_address = fake.address().replace('\n', ', ')
agreement_date = fake.date_between(start_date='-10y', end_date='today').strftime("%B %d, %Y")

print(f"Agreement Date: {agreement_date}")
print(f"Lender: {lender}, Address: {lender_address}")
print(f"Borrower: {borrower}, Address: {borrower_address}")

Agreement Date: July 07, 2025
Lender: Kim, Perry and Smith, Address: 19368 Craig Alley, Staceyside, ND 82505
Borrower: Fisher and Sons, Address: 0097 Archer Course Suite 202, Daletown, GU 70217


In [17]:
from faker import Faker
from transformers import AutoTokenizer, AutoModelForCausalLM

fake = Faker()

lender = fake.company()
lender_address = fake.address().replace('\n', ', ')
borrower = fake.company()
borrower_address = fake.address().replace('\n', ', ')
agreement_date = fake.date_between(start_date='-10y', end_date='today').strftime("%B %d, %Y")

topic = (
    f"Loan Agreement\n"
    f"Lender: {lender}, {lender_address}\n"
    f"Borrower: {borrower}, {borrower_address}\n"
    f"Agreement Date: {agreement_date}\n"
)

sections = [
    "0. INTRODUCTION",
    "1. PARTIES",
    "2. DEFINITIONS",
    "3. LENDING DISCLOSURE",
    "4. LOAN TERMS",
    "5. REPAYMENT TERMS",
    "6. INTEREST RATES AND FEES",
    "7. COLLATERAL",
    "8. COVENANTS",
    "9. DEFAULT AND REMEDIES",
    "10. MISCELLANEOUS PROVISIONS"
]

output_file = "synthetic/generated_contract.txt"
contract_so_far = ""  # This accumulates the contract's text as context.

with open(output_file, "w", encoding="utf-8") as f:
    f.write("=== SYNTHETIC CONTRACT GENERATION ===\n\n")
    f.write(f"{topic}\n")
    f.write("="*40 + "\n\n")

for section in sections:
    # Build the prompt using prior generated contract content for context (trim if very long!)
    # For small models, keep context window in mind (e.g. use "contract_so_far[-1500:]" for long contracts)
    prompt = (
        "You are a legal document drafting assistant.\n"
        "You need to generate a full, realistic financial agreement contract section by section. Each section should be coherent with the previous sections and follow legal language and style.\n"
        f"Content:{topic}\nContract so far: {contract_so_far[-1500:]}\n"  # Only use last ~1500 chars for context to avoid overflow.
        f"Generate only the text for the following section: {section}\n"
        # "Continue the contract with this section, following legal language and style.\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        inputs["input_ids"], max_new_tokens=400, do_sample=True, pad_token_id=tokenizer.eos_token_id
    )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Remove prompt echo (if present): get only the new text after the prompt
    if decoded.startswith(prompt):
        section_text = decoded[len(prompt):].strip()
    else:
        section_text = decoded.strip()

    # For the next iteration, accumulate the contract so far
    contract_so_far += f"\n\n{section}\n{section_text}"

    # Write to file with annotation
    with open(output_file, "a", encoding="utf-8") as f:
        f.write(f"\n\n--- Prompt for {section} ---\n")
        f.write(prompt.strip() + "\n")
        f.write(f"--- Output for {section} ---\n")
        f.write(section_text)
        f.write("\n" + "="*40 + "\n")

print(f"Done! See the generated contract in: {output_file}")

Done! See the generated contract in: synthetic/generated_contract.txt
